In [11]:
import os
import json
import shutil
from collections import Counter
from tqdm import tqdm

In [12]:
ROOT = os.path.abspath("..")

print("Project root:", ROOT)

TRAIN_ANNOS = os.path.join(ROOT, "data/raw/train/annos")
TRAIN_IMAGES = os.path.join(ROOT, "data/raw/train/images")

VAL_ANNOS = os.path.join(ROOT, "data/raw/validation/annos")
VAL_IMAGES = os.path.join(ROOT, "data/raw/validation/images")

OUT_TRAIN_ANNOS = os.path.join(ROOT, "data/processed/train/annos")
OUT_TRAIN_IMAGES = os.path.join(ROOT, "data/processed/train/images")

OUT_VAL_ANNOS = os.path.join(ROOT, "data/processed/validation/annos")
OUT_VAL_IMAGES = os.path.join(ROOT, "data/processed/validation/images")

os.makedirs(OUT_TRAIN_ANNOS, exist_ok=True)
os.makedirs(OUT_TRAIN_IMAGES, exist_ok=True)

os.makedirs(OUT_VAL_ANNOS, exist_ok=True)
os.makedirs(OUT_VAL_IMAGES, exist_ok=True)

print("\nPaths verified:")
print("TRAIN_ANNOS:", TRAIN_ANNOS)
print("TRAIN_IMAGES:", TRAIN_IMAGES)
print("VAL_ANNOS:", VAL_ANNOS)
print("VAL_IMAGES:", VAL_IMAGES)

Project root: /Users/harshsinha/Desktop/iiitbSemester/sem6/vr/project/dataset/miniProject

Paths verified:
TRAIN_ANNOS: /Users/harshsinha/Desktop/iiitbSemester/sem6/vr/project/dataset/miniProject/data/raw/train/annos
TRAIN_IMAGES: /Users/harshsinha/Desktop/iiitbSemester/sem6/vr/project/dataset/miniProject/data/raw/train/images
VAL_ANNOS: /Users/harshsinha/Desktop/iiitbSemester/sem6/vr/project/dataset/miniProject/data/raw/validation/annos
VAL_IMAGES: /Users/harshsinha/Desktop/iiitbSemester/sem6/vr/project/dataset/miniProject/data/raw/validation/images


In [13]:
category_counter = Counter()

for file in tqdm(os.listdir(TRAIN_ANNOS)):
    
    if not file.endswith(".json"):
        continue
    
    filepath = os.path.join(TRAIN_ANNOS, file)
    
    with open(filepath) as f:
        data = json.load(f)
    
    for key, value in data.items():
        
        if key.startswith("item"):
            
            category = value.get("category_name")
            
            if category:
                category_counter[category] += 1


top5 = category_counter.most_common(5)

print("Top 5 Categories:\n")

for cat, count in top5:
    print(f"{cat}: {count}")

TOP5 = set(cat for cat, _ in top5)

print("\nTOP5 set:", TOP5)

100%|██████████| 191961/191961 [00:28<00:00, 6713.46it/s]

Top 5 Categories:

short sleeve top: 71645
trousers: 55387
shorts: 36616
long sleeve top: 36064
skirt: 30835

TOP5 set: {'trousers', 'shorts', 'skirt', 'short sleeve top', 'long sleeve top'}


In [14]:
def filter_dataset(annos_dir, images_dir, out_annos, out_images):

    copied_images = 0
    missing_images = 0

    for file in tqdm(os.listdir(annos_dir)):

        if not file.endswith(".json"):
            continue

        json_path = os.path.join(annos_dir, file)

        with open(json_path) as f:
            data = json.load(f)

        filtered = {}

        for key, value in data.items():

            if key.startswith("item"):

                if value["category_name"] in TOP5:
                    filtered[key] = value

        if len(filtered) == 0:
            continue

        # keep metadata
        filtered["source"] = data.get("source")
        filtered["pair_id"] = data.get("pair_id")

        # save filtered annotation
        new_json = os.path.join(out_annos, file)

        with open(new_json, "w") as f:
            json.dump(filtered, f)

        # copy corresponding image
        image_name = file.replace(".json", ".jpg")

        src_img = os.path.join(images_dir, image_name)
        dst_img = os.path.join(out_images, image_name)

        if os.path.exists(src_img):

            if not os.path.exists(dst_img):
                shutil.copy(src_img, dst_img)

            copied_images += 1

        else:
            missing_images += 1

    print("\nImages copied:", copied_images)
    print("Missing images:", missing_images)

In [15]:
print("Filtering TRAIN dataset...")

# clear previous outputs (important if rerunning notebook)
if os.path.exists(OUT_TRAIN_ANNOS):
    shutil.rmtree(OUT_TRAIN_ANNOS)

if os.path.exists(OUT_TRAIN_IMAGES):
    shutil.rmtree(OUT_TRAIN_IMAGES)

os.makedirs(OUT_TRAIN_ANNOS, exist_ok=True)
os.makedirs(OUT_TRAIN_IMAGES, exist_ok=True)

filter_dataset(
    TRAIN_ANNOS,
    TRAIN_IMAGES,
    OUT_TRAIN_ANNOS,
    OUT_TRAIN_IMAGES
)

print("Train dataset filtered")

Filtering TRAIN dataset...


100%|██████████| 191961/191961 [02:07<00:00, 1503.66it/s]


Images copied: 144174
Missing images: 0
Train dataset filtered


In [16]:
print("Filtering VALIDATION dataset...")

# clear previous outputs
if os.path.exists(OUT_VAL_ANNOS):
    shutil.rmtree(OUT_VAL_ANNOS)

if os.path.exists(OUT_VAL_IMAGES):
    shutil.rmtree(OUT_VAL_IMAGES)

os.makedirs(OUT_VAL_ANNOS, exist_ok=True)
os.makedirs(OUT_VAL_IMAGES, exist_ok=True)

filter_dataset(
    VAL_ANNOS,
    VAL_IMAGES,
    OUT_VAL_ANNOS,
    OUT_VAL_IMAGES
)

print("Validation dataset filtered")

Filtering VALIDATION dataset...


100%|██████████| 32153/32153 [00:27<00:00, 1172.27it/s]


Images copied: 23741
Missing images: 0
Validation dataset filtered


In [17]:
import os

TRAIN_IMAGES = "../data/processed/train/images"
TRAIN_ANNOS = "../data/processed/train/annos"

missing = []

for file in os.listdir(TRAIN_ANNOS):
    
    if not file.endswith(".json"):
        continue
        
    img = file.replace(".json",".jpg")
    
    if not os.path.exists(os.path.join(TRAIN_IMAGES,img)):
        missing.append(img)

print("Missing images:",len(missing))

Missing images: 0


In [19]:
import os

IMG_DIR = "../data/raw/train/images"

ext_counter = {}

for file in os.listdir(IMG_DIR):
    ext = file.split(".")[-1]
    ext_counter[ext] = ext_counter.get(ext,0) + 1

print(ext_counter)

{'jpg': 191961}


In [20]:
import os

ROOT = "../data"

paths = [
    "raw/train",
    "raw/validation",
    "processed/train",
    "processed/validation"
]

for p in paths:
    
    full = os.path.join(ROOT, p)
    
    print(f"\nChecking: {full}")
    
    if not os.path.exists(full):
        print("❌ Folder does not exist")
        continue
        
    items = os.listdir(full)
    
    print("Contents:", items)
    
    if "annos" in items:
        annos = os.listdir(os.path.join(full,"annos"))
        print("annos count:", len(annos))
    else:
        print("❌ annos folder missing")
        
    if "images" in items:
        images = os.listdir(os.path.join(full,"images"))
        print("images count:", len(images))
    else:
        print("❌ images folder missing")


Checking: ../data/raw/train
Contents: ['.DS_Store', 'images', 'annos']
annos count: 191961
images count: 191961

Checking: ../data/raw/validation
Contents: ['.DS_Store', 'images', 'annos']
annos count: 32153
images count: 32153

Checking: ../data/processed/train
Contents: ['images', 'annos']
annos count: 144174
images count: 144174

Checking: ../data/processed/validation
Contents: ['images', 'annos']
annos count: 23741
images count: 23741
